# Hybrid Search RAG Pipeline: BM25 + MMR

Run this notebook with mrigi_tor190_v7


This notebook implements a **hybrid retrieval approach** that combines:
- **BM25 (keyword-based)**: Top 5 documents based on term frequency
- **MMR (semantic + diversity)**: Top 5 documents with diversity-aware vector search

**Final retrieval**: k=10 documents total (5 from each method, deduplicated)

Key improvements:
- Better coverage: Combines keyword matching with semantic similarity
- Increased diversity: MMR ensures diverse results
- Robust retrieval: Captures both exact term matches and semantic context

In [ ]:
import os
from pathlib import Path
os.environ["CUDA_VISIBLE_DEVICES"] = "3"
!source /home/jupyter/Mrigi/env.sh
hf_token = os.environ.get("HF_TOKEN")



import torch
# torch.cuda.init()
print(os.environ.get("CUDA_VISIBLE_DEVICES"))
torch.cuda.set_device(0)
import json

from langchain.embeddings import HuggingFaceEmbeddings
from langchain.embeddings import SentenceTransformerEmbeddings
from langchain.vectorstores import FAISS
from langchain.prompts import ChatPromptTemplate
from langchain.llms import HuggingFacePipeline
from langchain.chains import LLMChain
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from langchain.retrievers import BM25Retriever
from langchain.schema import Document

3


In [2]:
from pydantic import BaseModel, Field
from typing import Optional, Union
from datetime import datetime
from tqdm import tqdm

In [3]:
!nvidia-smi

Wed Nov 26 16:17:31 2025       
+-----------------------------------------------------------------------------+
| NVIDIA-SMI 495.29.05    Driver Version: 495.29.05    CUDA Version: 11.5     |
|-------------------------------+----------------------+----------------------+
| GPU  Name        Persistence-M| Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp  Perf  Pwr:Usage/Cap|         Memory-Usage | GPU-Util  Compute M. |
|                               |                      |               MIG M. |
|===============================+======================+======================|
|   0  NVIDIA RTX A5000    Off  | 00000000:31:00.0 Off |                  Off |
| 30%   31C    P8    21W / 230W |      8MiB / 24256MiB |      0%      Default |
|                               |                      |                  N/A |
+-------------------------------+----------------------+----------------------+
|   1  NVIDIA RTX A5000    Off  | 00000000:4B:00.0 Off |                  Off |
| 30%   

In [4]:

!kill -9 3964812

/bin/bash: line 0: kill: (3964812) - No such process


In [5]:
# ---------- Step 1: Load and Process the JSON Records ----------
# # Load JSON records from file
# with open('filtered_records_subset_100.json', 'r') as file:
#     filtered_records = json.load(file)

In [6]:
# Process each record: extract paragraphs with "Experimental" supersection,
# concatenate them, and append DOI info.
# documents = []
# metadata = []

In [7]:
# Process records into documents
# documents = []
# metadata = []

# for record in filtered_records:
#     doi = record.get("doi", "Unknown DOI")
    
#     # Add abstract as a separate document
#     abstract_text = record.get("abstract", "").strip()
#     if abstract_text:
#         combined_text = f"{abstract_text}\n\nThis information is from DOI: {doi}"
#         documents.append(combined_text)
#         metadata.append({"doi": doi, "source": "abstract"})
    
#     # Add each paragraph as a separate document
#     for para in record.get("paragraphs", []):
#         paragraph_text = para.get("text", "").strip()
#         if paragraph_text:
#             combined_text = f"{paragraph_text}\n\nThis information is from DOI: {doi}"
#             documents.append(combined_text)
#             metadata.append({"doi": doi, "source": "paragraph"})

# print(f"Total documents processed: {len(documents)}")

In [8]:
# for doc in documents:
#     print(doc)
#     print("-"*40)  # Optional: adds a visual separator between different papers


In [9]:
# ---------- Step 2: Prepare Embeddings for the Existing FAISS Index ----------
# Use the same model configuration that was used when the index was created.
# embeddings = HuggingFaceEmbeddings(model_name="allenai/scibert_scivocab_uncased")
embeddings = SentenceTransformerEmbeddings(model_name="all-MiniLM-L6-v2")

/tmp/ipykernel_443180/2006348050.py:4: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = SentenceTransformerEmbeddings(model_name="all-MiniLM-L6-v2")


In [10]:
# ---------- Step 2: Load the Existing Vector Database from Disk ----------
INDEX_DIRECTORY = Path("faiss_index")
INDEX_NAME = "index"
if not INDEX_DIRECTORY.exists():
    raise FileNotFoundError(f"Vector index directory '{INDEX_DIRECTORY}' not found.")

# Use allow_dangerous_deserialization=True to load indices saved with pickle-based metadata.
vector_db = FAISS.load_local(
    INDEX_DIRECTORY,
    embeddings,
    index_name=INDEX_NAME,
    allow_dangerous_deserialization=True
)
print(f"Loaded vector database from {INDEX_DIRECTORY} with {vector_db.index.ntotal} vectors")

Loaded vector database from faiss_index with 219350 vectors


In [11]:
# Configure a Max Marginal Relevance (MMR) retriever for diversity-aware retrieval
mmr_retriever = vector_db.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 5,           # final number of docs to return
        "fetch_k": 25,    # candidate pool size to trade off diversity/relevance
        "lambda_mult": 0.5  # 0 -> more diversity, 1 -> more relevance
    }
)

print("MMR retriever configured (k=5, fetch_k=25, lambda=0.5)")

MMR retriever configured (k=5, fetch_k=25, lambda=0.5)


In [12]:
# Initialize BM25 retriever for hybrid search
# First, get all documents from the vector database
all_docs = []
try:
    # Get documents directly from the docstore
    all_docs = list(vector_db.docstore._dict.values())
except Exception:
    # Fallback: iterate via index_to_docstore_id mapping
    ids_mapping = vector_db.index_to_docstore_id
    try:
        if isinstance(ids_mapping, dict):
            doc_ids = [ids_mapping[i] for i in range(vector_db.index.ntotal) if i in ids_mapping]
        else:
            doc_ids = list(ids_mapping)[: vector_db.index.ntotal]
        for doc_id in doc_ids:
            doc = vector_db.docstore.search(doc_id)
            if doc is not None:
                all_docs.append(doc)
    except Exception as e:
        print(f"Warning: Failed to collect documents for BM25: {e}")

# Initialize BM25 retriever with all documents
bm25_retriever = BM25Retriever.from_documents(all_docs)
bm25_retriever.k = 5  # Set to retrieve top 5 documents

print(f"BM25 retriever initialized with {len(all_docs)} documents (k=5)")

BM25 retriever initialized with 219350 documents (k=5)


In [13]:
# Hybrid search function: combines BM25 and MMR results
def hybrid_search(query: str, k_total=10, k_bm25=5, k_mmr=5, 
                  mmr_fetch_k=40, mmr_lambda=0.6):
    """
    Performs hybrid retrieval combining BM25 and MMR.
    
    Args:
        query: Search query
        k_total: Total number of documents to return (default: 10)
        k_bm25: Number of documents from BM25 (default: 5)
        k_mmr: Number of documents from MMR (default: 5)
        mmr_fetch_k: Fetch k parameter for MMR (default: 40)
        mmr_lambda: Lambda multiplier for MMR diversity (default: 0.6)
    
    Returns:
        List of Document objects (deduplicated if there's overlap)
    """
    # Get BM25 results
    bm25_retriever.k = k_bm25
    bm25_docs = bm25_retriever.get_relevant_documents(query)
    
    # Get MMR results
    mmr_docs = vector_db.max_marginal_relevance_search(
        query,
        k=k_mmr,
        fetch_k=mmr_fetch_k,
        lambda_mult=mmr_lambda
    )
    
    # Combine results and deduplicate based on content
    seen_contents = set()
    combined_docs = []
    
    # Add BM25 results first
    for doc in bm25_docs:
        content_hash = doc.page_content[:200]  # Use first 200 chars as identifier
        if content_hash not in seen_contents:
            seen_contents.add(content_hash)
            combined_docs.append(doc)
    
    # Add MMR results (only if not already included)
    for doc in mmr_docs:
        content_hash = doc.page_content[:200]
        if content_hash not in seen_contents:
            seen_contents.add(content_hash)
            combined_docs.append(doc)
    
    # Limit to k_total documents
    combined_docs = combined_docs[:k_total]
    
    return combined_docs

print("Hybrid search function defined (combines BM25 + MMR)")

Hybrid search function defined (combines BM25 + MMR)


In [14]:
# # Print DOIs for up to the first 50 processed documents
# if not metadata:
#     print("No metadata available. Did you load the records?")
# else:
#     for entry in metadata[:50]:
#         print(entry.get("doi", "Unknown DOI"))

In [15]:
# Get all documents and their metadata from the vector database WITHOUT running a similarity search
# This avoids embedding a query (and thus avoids GPU/cuBLAS initialization issues)

docs = []

# Fast path: if the docstore is an in-memory dict, grab all documents directly
try:
    # langchain InMemoryDocstore exposes a private _dict that maps id -> Document
    docs = list(vector_db.docstore._dict.values())
except Exception:
    # Fallback: iterate via index_to_docstore_id mapping
    ids_mapping = vector_db.index_to_docstore_id
    try:
        if isinstance(ids_mapping, dict):
            doc_ids = [ids_mapping[i] for i in range(vector_db.index.ntotal) if i in ids_mapping]
        else:
            # list-like mapping
            doc_ids = list(ids_mapping)[: vector_db.index.ntotal]
        for doc_id in doc_ids:
            doc = vector_db.docstore.search(doc_id)
            if doc is not None:
                docs.append(doc)
    except Exception as e:
        raise RuntimeError(f"Failed to collect documents from FAISS docstore: {e}")

# Compute unique DOIs
unique_dois = set()
for doc in docs:
    md = getattr(doc, 'metadata', {}) or {}
    unique_dois.add(md.get('doi', 'Unknown DOI'))

print(f"Number of documents: {len(docs)}")
print(f"Number of unique DOIs: {len(unique_dois)}")
print("\nUnique DOIs:")
for doi in sorted(unique_dois):
    print(doi)

Number of documents: 219350
Number of unique DOIs: 6228

Unique DOIs:
10.1002/(sici)1097-461x(1999)75:4/5<725::aid-qua39>3.0.co;2-i
10.1002/(sici)1097-4660(199603)65:3<221::aid-jctb402>3.0.co;2-y
10.1002/(sici)1097-4660(199603)65:3<265::aid-jctb413>3.0.co;2-e
10.1002/(sici)1097-4660(199704)68:4<432::aid-jctb617>3.0.co;2-x
10.1002/(sici)1097-4660(199705)69:1<27::aid-jctb682>3.0.co;2-j
10.1002/(sici)1097-4660(199801)71:1<6::aid-jctb807>3.0.co;2-d
10.1002/(sici)1097-4660(199812)73:4<377::aid-jctb973>3.0.co;2-i
10.1002/(sici)1097-4660(200004)75:4<279::aid-jctb217>3.0.co;2-a
10.1002/(sici)1099-0518(199603)34:4<673::aid-pola14>3.0.co;2-l
10.1002/(sici)1099-0682(199903)1999:3<361::aid-ejic361>3.0.co;2-p
10.1002/(sici)1099-0690(200002)2000:3<473::aid-ejoc473>3.0.co;2-n
10.1002/(sici)1099-1018(199605)20:3<145::aid-fam569>3.0.co;2-l
10.1002/(sici)1099-1581(199711)8:11<641::aid-pat694>3.0.co;2-a
10.1002/(sici)1521-3749(200006)626:6<1460::aid-zaac1460>3.0.co;2-x
10.1002/(sici)1521-3773(19980202)37

In [16]:
import pandas as pd

# Read the CSV file
zeosyn_df = pd.read_csv('ZEOSYN.csv')

# Get list of DOIs from the CSV
zeosyn_dois = set(zeosyn_df['doi'].unique())

# Find overlapping DOIs
overlapping_dois = unique_dois.intersection(zeosyn_dois)

# Print the count
print(f"Number of overlapping DOIs: {len(overlapping_dois)}")

# Save overlapping DOIs to variable
overlapping_dois_list = list(overlapping_dois)

Number of overlapping DOIs: 400


/tmp/ipykernel_443180/741441089.py:4: DtypeWarning: Columns (63,73) have mixed types. Specify dtype option on import or set low_memory=False.
  zeosyn_df = pd.read_csv('ZEOSYN.csv')


In [17]:
# Filter zeosyn_df to only include overlapping DOIs
filtered_df = zeosyn_df[zeosyn_df['doi'].isin(overlapping_dois_list)]

# Find DOIs where Code1 is empty (NaN)
empty_code1_dois = filtered_df[filtered_df['Code1'].isna()]['doi'].unique()
print(f"Number of DOIs with empty Code1: {len(empty_code1_dois)}")
print("\nDOIs with empty Code1:")
for doi in empty_code1_dois:
    print(doi)

# Get unique Code1 entries for overlapping DOIs
unique_codes = filtered_df['Code1'].dropna().unique()
print(f"\nNumber of unique Code1 entries: {len(unique_codes)}")
print("\nUnique Code1 entries:")
for code in sorted(unique_codes):
    print(code)

# Store unique codes in a variable
unique_code1_entries = sorted(unique_codes)

Number of DOIs with empty Code1: 102

DOIs with empty Code1:
10.1002/anie.201404608
10.1002/anie.201900106
10.1002/chem.201504717
10.1002/chem.201704668
10.1002/chem.201802087
10.1002/chem.201802255
10.1007/s10562-015-1683-4
10.1007/s11244-010-9591-8
10.1016/j.cattod.2018.06.011
10.1016/j.micromeso.2006.02.021
10.1016/j.micromeso.2009.02.027
10.1016/j.micromeso.2009.08.021
10.1016/j.micromeso.2010.09.034
10.1016/j.micromeso.2011.03.023
10.1016/j.micromeso.2012.04.022
10.1016/j.micromeso.2013.12.013
10.1016/j.micromeso.2014.04.002
10.1016/j.micromeso.2015.07.022
10.1016/j.micromeso.2015.11.007
10.1016/j.micromeso.2016.08.025
10.1016/j.micromeso.2016.08.039
10.1016/j.micromeso.2019.03.051
10.1016/j.micromeso.2020.110027
10.1016/s1387-1811(98)00314-x
10.1021/ic201515z
10.1039/c7dt00640c
10.1002/chem.201804973
10.1016/j.micromeso.2005.06.023
10.1016/j.micromeso.2016.06.026
10.1016/s1387-1811(03)00466-9
10.1016/s1387-1811(99)00138-9
10.1021/cm071753o
10.1021/cm991057r
10.1039/c7ta10002g
10.

In [18]:
# Save overlapping DOIs minus those with empty Code1 to a JSON file

timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
output_file = Path(f"overlapping_dois_minus_empty_code1_{timestamp}.json")

# Ensure overlapping_dois_list exists
if 'overlapping_dois_list' not in globals():
    raise RuntimeError("Variable 'overlapping_dois_list' not found. Run the cell that computes overlapping DOIs first.")

overlapping_set = set(overlapping_dois_list)

# Use existing empty_code1_dois if available, otherwise compute it
if 'empty_code1_dois' in globals():
    empty_set = set(empty_code1_dois)
else:
    empty_set = set(
        zeosyn_df[
            (zeosyn_df['doi'].isin(overlapping_dois_list)) &
            (zeosyn_df['Code1'].isna())
        ]['doi'].unique()
    )

filtered_dois = sorted(list(overlapping_set - empty_set))

# Save to JSON
with open(output_file, 'w', encoding='utf-8') as f:
    json.dump(filtered_dois, f, indent=2, ensure_ascii=False)

print(f"Saved {len(filtered_dois)} DOIs to {output_file}")
print("First 10 DOIs (sample):")
for doi in filtered_dois[:10]:
    print(doi)

Saved 298 DOIs to overlapping_dois_minus_empty_code1_20251126_161748.json
First 10 DOIs (sample):
10.1002/1099-0682(200105)2001:5<1167::aid-ejic1167>3.0.co;2-z
10.1002/1521-3773(20010618)40:12<2277::aid-anie2277>3.0.co;2-o
10.1002/admi.202002029
10.1002/aic.10217
10.1002/aic.17177
10.1002/anie.198900811
10.1002/anie.200461911
10.1002/anie.200503011
10.1002/anie.200803578
10.1002/anie.201309766


In [19]:
# Create a dictionary to store Code1 -> DOIs mapping
code1_doi_mapping = {}

# Filter zeosyn_df to only include rows where:
# 1. doi is in overlapping_dois_list
# 2. Code1 is not null
filtered_df = zeosyn_df[
    (zeosyn_df['doi'].isin(overlapping_dois_list)) & 
    (zeosyn_df['Code1'].notna())
]

# Group by Code1 and collect unique DOIs for each code
for code in unique_code1_entries:
    code_dois = filtered_df[filtered_df['Code1'] == code]['doi'].unique().tolist()
    if code_dois:  # Only add to mapping if there are matching DOIs
        code1_doi_mapping[code] = code_dois

# Print summary
print(f"Created mapping for {len(code1_doi_mapping)} unique Code1 entries")
print(f"Sample of mapping (first 3 entries):")
for code, dois in list(code1_doi_mapping.items())[:3]:
    print(f"\n{code}:")
    print(f"Number of DOIs: {len(dois)}")
    print(f"Example DOIs: {dois[:2]}")

Created mapping for 113 unique Code1 entries
Sample of mapping (first 3 entries):

*-SVY:
Number of DOIs: 2
Example DOIs: ['10.1039/c7dt00640c', '10.1016/j.cattod.2019.03.006']

*BEA:
Number of DOIs: 37
Example DOIs: ['10.1002/chem.201504717', '10.1016/j.micromeso.2006.02.021']

*CTH:
Number of DOIs: 2
Example DOIs: ['10.1002/anie.201912488', '10.1021/acs.chemmater.7b01181']


In [20]:
# Create a modified mapping by ignoring decorators (* and -) in Code1
clean_code1_doi_mapping = {}

# Function to clean framework code by removing * and -
def clean_framework_code(code):
    return code.replace('*', '').replace('-', '') if isinstance(code, str) else code

# Filter zeosyn_df to only include rows where:
# 1. doi is in overlapping_dois_list  
# 2. Code1 is not null
filtered_df = zeosyn_df[
    (zeosyn_df['doi'].isin(overlapping_dois_list)) & 
    (zeosyn_df['Code1'].notna())
]

# Group by cleaned Code1 and collect unique DOIs for each code
for _, row in filtered_df.iterrows():
    code = row['Code1']
    clean_code = clean_framework_code(code)
    if clean_code:
        doi = row['doi']
        if clean_code not in clean_code1_doi_mapping:
            clean_code1_doi_mapping[clean_code] = []
        if doi not in clean_code1_doi_mapping[clean_code]:
            clean_code1_doi_mapping[clean_code].append(doi)

# Print summary
print(f"Created mapping for {len(clean_code1_doi_mapping)} unique cleaned Code1 entries")
print(f"\nSample of mapping (first 3 entries):")
for code, dois in list(clean_code1_doi_mapping.items())[:3]:
    print(f"\n{code}:")
    print(f"Number of DOIs: {len(dois)}")
    print(f"Example DOIs: {dois[:2]}")

Created mapping for 113 unique cleaned Code1 entries

Sample of mapping (first 3 entries):

SOS:
Number of DOIs: 1
Example DOIs: ['10.1002/anie.200461911']

JRY:
Number of DOIs: 1
Example DOIs: ['10.1002/anie.200803578']

POS:
Number of DOIs: 1
Example DOIs: ['10.1002/anie.201309766']


In [21]:
# Save framework DOI mapping to JSON.
# Behavior:
# - If a variable named `framework` or `selected_framework` exists and matches a key in the mapping,
#   save only that framework's entry.
# - Otherwise save the entire cleaned mapping.
# Requires: `clean_code1_doi_mapping` or `code1_doi_mapping` to be present in the notebook.

mapping = globals().get('clean_code1_doi_mapping') or globals().get('code1_doi_mapping')
if mapping is None:
    raise RuntimeError("No mapping found. Ensure 'clean_code1_doi_mapping' or 'code1_doi_mapping' is defined.")

# Determine single-framework save if requested
framework_key = None
if 'framework' in globals() and isinstance(framework := globals()['framework'], str):
    framework_key = framework
elif 'selected_framework' in globals() and isinstance(selected_framework := globals()['selected_framework'], str):
    framework_key = selected_framework

timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
output_dir = Path("outputs")
output_dir.mkdir(parents=True, exist_ok=True)

if framework_key and framework_key in mapping:
    out_data = {framework_key: mapping[framework_key]}
    out_file = output_dir / f"{framework_key}_dois_{timestamp}.json"
else:
    out_data = mapping
    out_file = output_dir / f"clean_code1_doi_mapping_{timestamp}.json"

with open(out_file, 'w', encoding='utf-8') as f:
    json.dump(out_data, f, indent=2, ensure_ascii=False)

print(f"Saved {len(out_data) if isinstance(out_data, dict) else 0} entries to {out_file}")

Saved 113 entries to outputs/clean_code1_doi_mapping_20251126_161748.json


In [22]:
# Function to clean framework code by removing * and -
def clean_framework_code(code):
    return code.replace('*', '').replace('-', '') if isinstance(code, str) else code

# Create a new list of unique cleaned codes
cleaned_codes = set()
for code in unique_code1_entries:
    clean_code = clean_framework_code(code)
    if clean_code:
        cleaned_codes.add(clean_code)

# Convert back to sorted list
cleaned_unique_codes = sorted(list(cleaned_codes))

print(f"Original number of unique codes: {len(unique_code1_entries)}")
print(f"Number of unique codes after cleaning: {len(cleaned_unique_codes)}")
print("\nFirst 10 cleaned codes:")
for code in cleaned_unique_codes[:10]:
    print(code)

Original number of unique codes: 113
Number of unique codes after cleaning: 113

First 10 cleaned codes:
ABW
ACO
AEI
AEL
AEN
AFI
AFN
AFS
AFX
ANA


In [23]:
# Create a list of queries for each framework
queries = []
for code in cleaned_unique_codes:
    query = f"Extract the following synthesis information for {code} or {code}-like frameworks: [crytallization temeperature], [crystallization time], [precursor1, precursor2, ..], [OSDA1, OSDA2, etc]"

    queries.append(query)

# Print first few queries as example
print("First 5 queries:")
for q in queries[:5]:
    print(q)

print(f"\nTotal number of queries generated: {len(queries)}")

First 5 queries:
Extract the following synthesis information for ABW or ABW-like frameworks: [crytallization temeperature], [crystallization time], [precursor1, precursor2, ..], [OSDA1, OSDA2, etc]
Extract the following synthesis information for ACO or ACO-like frameworks: [crytallization temeperature], [crystallization time], [precursor1, precursor2, ..], [OSDA1, OSDA2, etc]
Extract the following synthesis information for AEI or AEI-like frameworks: [crytallization temeperature], [crystallization time], [precursor1, precursor2, ..], [OSDA1, OSDA2, etc]
Extract the following synthesis information for AEL or AEL-like frameworks: [crytallization temeperature], [crystallization time], [precursor1, precursor2, ..], [OSDA1, OSDA2, etc]
Extract the following synthesis information for AEN or AEN-like frameworks: [crytallization temeperature], [crystallization time], [precursor1, precursor2, ..], [OSDA1, OSDA2, etc]

Total number of queries generated: 113


In [24]:
# ---------- Step 3: Use the Vector DB in a RAG Pipeline ----------
# Define your query.
# query = "How is hierarchical ZSM-5 synthesized?"
# query = "What is the best way to synthesize ZSM-5 that is hierarchical?"
# query = "How is Silicalite-1 synthesized?"
query = "ABW zeolite synthesis conditions,synthesis of ABW framework, ABW-type zeolite preparation method, hydrothermal synthesis of ABW zeolite"


In [25]:
# Retrieve documents using hybrid search (BM25 + MMR)
retrieved_docs = hybrid_search(query, k_total=10, k_bm25=5, k_mmr=5)
context_text = "\n\n".join([doc.page_content for doc in retrieved_docs])
print(f"Hybrid search retrieved {len(retrieved_docs)} docs (5 from BM25, 5 from MMR)")

/tmp/ipykernel_443180/2734919117.py:20: LangChainDeprecationWarning: The method `BaseRetriever.get_relevant_documents` was deprecated in langchain-core 0.1.46 and will be removed in 1.0. Use :meth:`~invoke` instead.
  bm25_docs = bm25_retriever.get_relevant_documents(query)


Hybrid search retrieved 10 docs (5 from BM25, 5 from MMR)


In [26]:
# Diagnostic comparison: cosine vs MMR overlap for current query
try:
    baseline_docs = vector_db.similarity_search(query, k=5)
    baseline_set = {d.page_content[:150] for d in baseline_docs}
    mmr_set = {d.page_content[:150] for d in retrieved_docs}
    overlap = len(baseline_set & mmr_set)
    print(f"Overlap between cosine top-5 and MMR top-5: {overlap}")
    print(f"Unique to MMR: {len(mmr_set - baseline_set)} | Unique to cosine: {len(baseline_set - mmr_set)}")
except Exception as e:
    print(f"Diagnostic comparison failed: {e}")

Overlap between cosine top-5 and MMR top-5: 1
Unique to MMR: 9 | Unique to cosine: 2


In [27]:
# Diagnostic: Analyze composition of hybrid search results
try:
    # Get individual BM25 and MMR results for comparison
    bm25_retriever.k = 5
    bm25_only = bm25_retriever.get_relevant_documents(query)
    mmr_only = vector_db.max_marginal_relevance_search(query, k=5, fetch_k=40, lambda_mult=0.6)
    
    # Create sets for comparison (using first 150 chars as identifier)
    bm25_set = {d.page_content[:150] for d in bm25_only}
    mmr_set = {d.page_content[:150] for d in mmr_only}
    hybrid_set = {d.page_content[:150] for d in retrieved_docs}
    
    # Calculate overlaps
    bm25_mmr_overlap = len(bm25_set & mmr_set)
    bm25_in_hybrid = len(bm25_set & hybrid_set)
    mmr_in_hybrid = len(mmr_set & hybrid_set)
    
    print(f"\n=== Hybrid Search Analysis ===")
    print(f"BM25-MMR overlap: {bm25_mmr_overlap}/5 documents")
    print(f"BM25 docs in hybrid result: {bm25_in_hybrid}/5")
    print(f"MMR docs in hybrid result: {mmr_in_hybrid}/5")
    print(f"Total unique docs in hybrid: {len(hybrid_set)}")
    print(f"Unique to BM25: {len(bm25_set - mmr_set)}")
    print(f"Unique to MMR: {len(mmr_set - bm25_set)}")
except Exception as e:
    print(f"Diagnostic analysis failed: {e}")


=== Hybrid Search Analysis ===
BM25-MMR overlap: 0/5 documents
BM25 docs in hybrid result: 5/5
MMR docs in hybrid result: 5/5
Total unique docs in hybrid: 10
Unique to BM25: 5
Unique to MMR: 5


In [28]:
# Initialize lists to store results for each query
all_retrieved_docs = []
all_context_texts = []

# Define hybrid search parameters
hybrid_k_total = 10  # Total documents to retrieve
hybrid_k_bm25 = 5    # Documents from BM25
hybrid_k_mmr = 5     # Documents from MMR
mmr_fetch_k = 40     # Fetch k for MMR
mmr_lambda = 0.6     # Lambda for MMR diversity

# Process each query with hybrid search (BM25 + MMR)
for q in tqdm(queries, desc="Processing queries (Hybrid: BM25 + MMR)"):
    docs = hybrid_search(
        q, 
        k_total=hybrid_k_total, 
        k_bm25=hybrid_k_bm25, 
        k_mmr=hybrid_k_mmr,
        mmr_fetch_k=mmr_fetch_k,
        mmr_lambda=mmr_lambda
    )
    all_retrieved_docs.append(docs)
    context = "\n\n".join([doc.page_content for doc in docs])
    all_context_texts.append(context)

print(f"Processed {len(queries)} queries with hybrid search (BM25 + MMR)")
print(f"Parameters: k_total={hybrid_k_total}, k_bm25={hybrid_k_bm25}, k_mmr={hybrid_k_mmr}")
print(f"MMR params: fetch_k={mmr_fetch_k}, lambda={mmr_lambda}")
print(f"Retrieved {len(all_retrieved_docs)} sets of documents")
print(f"Generated {len(all_context_texts)} context texts")

Processing queries (Hybrid: BM25 + MMR): 100%|██████████| 113/113 [02:33<00:00,  1.36s/it]

Processed 113 queries with hybrid search (BM25 + MMR)
Parameters: k_total=10, k_bm25=5, k_mmr=5
MMR params: fetch_k=40, lambda=0.6
Retrieved 113 sets of documents
Generated 113 context texts


In [29]:
# Hybrid search parameter configuration
hybrid_k_total = 10  # Total documents per query
hybrid_k_bm25 = 5    # Documents from BM25 retrieval
hybrid_k_mmr = 5     # Documents from MMR retrieval
mmr_fetch_k = 40     # Candidate pool for MMR
mmr_lambda = 0.6     # Trade-off parameter (lower -> more diversity)

print(f"Hybrid search params set:")
print(f"  Total k: {hybrid_k_total} (BM25: {hybrid_k_bm25}, MMR: {hybrid_k_mmr})")
print(f"  MMR params: fetch_k={mmr_fetch_k}, lambda={mmr_lambda}")

Hybrid search params set:
  Total k: 10 (BM25: 5, MMR: 5)
  MMR params: fetch_k=40, lambda=0.6


In [30]:
# print(embeddings.embed_query("Silicalite-1 synthesis"))
# print(embeddings.embed_documents(["test text 1", "test text 2"]))


In [31]:
context_text

"An analysis of the NaZnPO4-ABW structure with the program KRIBER revealed that the tetrahedral topology (coordination sequences) of this phase is identical to those of the ABW and ATN frameworks. The ABW topology has been extensively discussed previously, and may variously be regarded as being built up from zigzag chains or six-ring sheets of tetrahedra. NaZnPO4-ABW is another example of the MABO4 class of ABW-type zeolite isotypes. It complements the previously described lithium zinc phosphate hydrate, LiZnPO4*H2O, which is an example of the other (LiABO4*H2O) class of ABW-type structure. NaZnPO4-ABW is the first reported anhydrous ABW isostructure to contain sodium cations, rather than larger univalent species.\n\nThis information is from DOI: 10.1016/s1387-1811(98)00111-5\n\nIn this paper, we report the solution-mediated synthesis and single-crystal structure of NaZnPO4-ABW, a new monoclinic modification of the ABW structure type containing highly elliptical eight-rings. NaZnPO4-AB

In [32]:
# RAG prompt template
RAG_PROMPT = """
Answer the question based only on the following context:
{context}
Question: {question}
Provide a detailed answer. Tell the user you can't answer the question if the context does not have anything useful to answer the question.
Provide which DOI the answer is retrieved from.
"""

# # Judge prompt template for evaluation
# JUDGE_PROMPT = """
# You are an expert judge evaluating the quality of an answer generated by an AI system.
# You will be provided with:
# 1. The original question
# 2. The context provided to the AI
# 3. The AI's answer

# Please evaluate the answer on the following criteria:
# 1. Relevance (0-10): How well does the answer address the question?
# 2. Accuracy (0-10): How accurate is the answer based on the provided context?
# 3. Completeness (0-10): How complete is the answer?
# 4. Citation (0-10): Does it properly cite the DOI?

# Question: {question}
# Context: {context}
# Answer: {answer}

# Return ONLY a valid JSON object with no additional text or prefixes:
# {{
#     "relevance": {{"score": 8, "explanation": "Brief explanation here"}}, 
#     "accuracy": {{"score": 9, "explanation": "Brief explanation here"}},
#     "completeness": {{"score": 7, "explanation": "Brief explanation here"}},
#     "citation": {{"score": 10, "explanation": "Brief explanation here"}},
#     "total_score": 34,
#     "overall_feedback": "Overall assessment here"
# }}
# """

In [33]:
rag_prompt = ChatPromptTemplate.from_template(RAG_PROMPT)
# judge_prompt = ChatPromptTemplate.from_template(JUDGE_PROMPT)
prompt = rag_prompt.format(context=context_text, question=query)

In [34]:
# Create a list of tuples containing (code, prompt, context)
prompts_with_codes = []
for code, query, context in zip(cleaned_unique_codes, queries, all_context_texts):
    formatted_prompt = rag_prompt.format(context=context, question=query)
    prompts_with_codes.append({
        'code': code,
        'query': query,
        'prompt': formatted_prompt
    })

print(f"Created {len(prompts_with_codes)} prompt-code pairs")
print("\nExample for first entry:")
print(f"Code: {prompts_with_codes[0]['code']}")
print(f"Query: {prompts_with_codes[0]['query']}")
print(f"Prompt length: {len(prompts_with_codes[0]['prompt'])}")

Created 113 prompt-code pairs

Example for first entry:
Code: ABW
Query: Extract the following synthesis information for ABW or ABW-like frameworks: [crytallization temeperature], [crystallization time], [precursor1, precursor2, ..], [OSDA1, OSDA2, etc]
Prompt length: 5923


In [35]:
# # Create list of prompts for all queries and their contexts
# rag_prompts = []

# # Iterate through queries and contexts
# for query, context in zip(queries, all_context_texts):
#     # Format prompt for each query-context pair using the existing template
#     formatted_prompt = rag_prompt.format(context=context, question=query)
#     rag_prompts.append(formatted_prompt)

# print(f"Created {len(rag_prompts)} prompts")
# print("\nExample of first prompt:")
# print(rag_prompts[0])

In [36]:
# ---------- Step 4: Generate an Answer Using an Open-Source LLM ----------
# Setup the open source LLM using Mistral (ensure you have the model or access to it)
model_name = "mistralai/Mistral-7B-Instruct-v0.1"

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True, use_fast=False, use_auth_token=hf_token)

/home/synthesisproject/miniforge3/envs/mrigi_tor190_v7/lib/python3.9/site-packages/transformers/models/auto/tokenization_auto.py:655: FutureWarning: The `use_auth_token` argument is deprecated and will be removed in v5 of Transformers.
  warnings.warn(
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


In [38]:
# from transformers.models.auto.configuration_auto import CONFIG_MAPPING
# print("Supported model types:", list(CONFIG_MAPPING.keys()))

In [ ]:
model = AutoModelForCausalLM.from_pretrained(model_name, device_map="auto", trust_remote_code=True, use_auth_token=hf_token, torch_dtype=torch.float16)

/home/synthesisproject/miniforge3/envs/mrigi_tor190_v7/lib/python3.9/site-packages/transformers/models/auto/auto_factory.py:472: FutureWarning: The `use_auth_token` argument is deprecated and will be removed in v5 of Transformers.
  warnings.warn(


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

/home/synthesisproject/miniforge3/envs/mrigi_tor190_v7/lib/python3.9/site-packages/transformers/utils/hub.py:374: FutureWarning: The `use_auth_token` argument is deprecated and will be removed in v5 of Transformers.
  warnings.warn(


In [40]:
hf_pipeline = pipeline("text-generation", model=model, tokenizer=tokenizer, max_length=10000)

In [41]:
# Wrap the Hugging Face pipeline with LangChain's HuggingFacePipeline interface.
rag_llm = HuggingFacePipeline(pipeline=hf_pipeline)

# Initialize separate judge pipeline (using same model but shorter max length for efficiency)
# judge_pipeline = pipeline("text-generation", model=model, tokenizer=tokenizer, max_length=2000)
# judge_llm = HuggingFacePipeline(pipeline=judge_pipeline)

/tmp/ipykernel_443180/960563000.py:2: LangChainDeprecationWarning: The class `HuggingFacePipeline` was deprecated in LangChain 0.0.37 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFacePipeline``.
  rag_llm = HuggingFacePipeline(pipeline=hf_pipeline)


In [42]:
# Generate the answer based on the prompt that includes retrieved context.
response_text = rag_llm(prompt)
print("Response:")
print(response_text)

/tmp/ipykernel_443180/4206892105.py:2: LangChainDeprecationWarning: The method `BaseLLM.__call__` was deprecated in langchain-core 0.1.7 and will be removed in 1.0. Use :meth:`~invoke` instead.
  response_text = rag_llm(prompt)
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Response:
Human: 
Answer the question based only on the following context:
An analysis of the NaZnPO4-ABW structure with the program KRIBER revealed that the tetrahedral topology (coordination sequences) of this phase is identical to those of the ABW and ATN frameworks. The ABW topology has been extensively discussed previously, and may variously be regarded as being built up from zigzag chains or six-ring sheets of tetrahedra. NaZnPO4-ABW is another example of the MABO4 class of ABW-type zeolite isotypes. It complements the previously described lithium zinc phosphate hydrate, LiZnPO4*H2O, which is an example of the other (LiABO4*H2O) class of ABW-type structure. NaZnPO4-ABW is the first reported anhydrous ABW isostructure to contain sodium cations, rather than larger univalent species.

This information is from DOI: 10.1016/s1387-1811(98)00111-5

In this paper, we report the solution-mediated synthesis and single-crystal structure of NaZnPO4-ABW, a new monoclinic modification of the A

In [43]:
# ...existing code...
import json
from datetime import datetime
from pathlib import Path

# Create a timestamp for the filename
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
output_file = f'framework_synthesis_responses_{timestamp}.json'

# Initialize responses dict
responses_data = {}

# Process each prompt and save results incrementally
for item in tqdm(prompts_with_codes, desc="Generating responses"):
    try:
        # Get the pre-formatted prompt string
        prompt_text = item.get('prompt', '')
        if not isinstance(prompt_text, str) or not prompt_text.strip():
            raise ValueError("Missing or empty formatted prompt in prompts_with_codes")

        # Generate response using the formatted prompt
        response = rag_llm.invoke(prompt_text)

        # Ensure response is a serializable string
        if isinstance(response, (dict, list)):
            response_text = json.dumps(response, ensure_ascii=False)
        else:
            response_text = str(response)

        # Create entry for this framework
        code_val = item.get('code', 'UNKNOWN')
        responses_data[code_val] = {
            'code': code_val,
            'query': item.get('query', ''),
            'prompt': prompt_text,
            'response': response_text,
            'timestamp': datetime.now().isoformat()
        }

        # Save updated results after each successful generation
        with open(output_file, 'w', encoding='utf-8') as f:
            json.dump(responses_data, f, indent=2, ensure_ascii=False)

    except Exception as e:
        print(f"Error processing {item.get('code', 'UNKNOWN')}: {str(e)}")
        continue

print(f"Completed processing {len(responses_data)} frameworks")
print(f"Results saved to {output_file}")
# ...existing code...

Generating responses:   0%|          | 0/113 [00:00<?, ?it/s]Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Generating responses:   1%|          | 1/113 [00:02<04:12,  2.25s/it]Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Generating responses:   2%|▏         | 2/113 [00:04<03:37,  1.96s/it]Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Generating responses:   4%|▎         | 4/113 [00:07<03:08,  1.73s/it]Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Generating responses:   4%|▍         | 5/113 [00:08<02:58,  1.65s/it]Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open

Completed processing 113 frameworks
Results saved to framework_synthesis_responses_20251126_162048.json


# LLM Judge Evaluation System

In [44]:
# def evaluate_rag_response(query, k=2):
#     # Get relevant documents
#     retrieved_docs = vector_db.similarity_search(query, k=k)
#     context_text = "\n\n".join([doc.page_content for doc in retrieved_docs])
    
#     # Generate RAG response
#     rag_chain = LLMChain(llm=rag_llm, prompt=rag_prompt)
#     response = rag_chain.run(context=context_text, question=query)
    
#     # Generate evaluation
#     judge_chain = LLMChain(llm=judge_llm, prompt=judge_prompt)
#     evaluation = judge_chain.run(
#         question=query,
#         context=context_text,
#         answer=response
#     )
    
#     # Parse the evaluation with improved JSON extraction
#     try:
#         # Try to extract JSON from the response
#         import re
        
#         # Look for JSON object starting with { and ending with }
#         json_match = re.search(r'\{.*\}', evaluation, re.DOTALL)
#         if json_match:
#             json_str = json_match.group(0)
#             evaluation_dict = json.loads(json_str)
#         else:
#             # If no JSON found, try parsing the whole string
#             evaluation_dict = json.loads(evaluation)
            
#     except (json.JSONDecodeError, AttributeError) as e:
#         evaluation_dict = {
#             "error": "Failed to parse evaluation",
#             "raw_evaluation": evaluation,
#             "parse_error": str(e)
#         }
    
#     return {
#         "query": query,
#         "context": context_text,
#         "response": response,
#         "evaluation": evaluation_dict,
#         "timestamp": datetime.now().isoformat()
#     }

In [45]:
# # List of test queries - using just the MFI alkali query for now
# test_queries = [
#     "How can you study theeffect of nature of silica source on the purity of template-free ZSM-5?"
# ]

# # Run evaluation for each query
# results = []
# for query in tqdm(test_queries, desc="Evaluating queries"):
#     result = evaluate_rag_response(query)
#     results.append(result)

# # Save results to JSON file
# output_filename = f"rag_evaluation_results_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
# with open(output_filename, 'w') as f:
#     json.dump(results, f, indent=2)

# print(f"Results saved to {output_filename}")

In [46]:
# # Calculate average scores
# def analyze_results(results):
#     total_scores = {
#         'relevance': 0,
#         'accuracy': 0,
#         'completeness': 0,
#         'citation': 0,
#         'total_score': 0
#     }
#     valid_results = 0
    
#     for result in results:
#         eval_dict = result['evaluation']
#         if 'error' not in eval_dict:
#             valid_results += 1
#             total_scores['relevance'] += eval_dict['relevance']['score']
#             total_scores['accuracy'] += eval_dict['accuracy']['score']
#             total_scores['completeness'] += eval_dict['completeness']['score']
#             total_scores['citation'] += eval_dict['citation']['score']
#             total_scores['total_score'] += eval_dict['total_score']
    
#     if valid_results > 0:
#         avg_scores = {k: v/valid_results for k, v in total_scores.items()}
#         print("\nAverage Scores:")
#         for metric, score in avg_scores.items():
#             print(f"{metric}: {score:.2f}")
#     else:
#         print("No valid evaluations found")

# analyze_results(results)